# Recurrent Neural NetworkによるBitcoinの価格予測

---
## 目的
Recurrent Neural Networkを使ってBitcoinの価格予測を行う．リカレントニューラルネットワークの構造や学習方法については`rnn.ipynb`で説明したものと同様のため，ここでは省略する．
また，CSVファイルなどのテキストファイルで整理・保存された数値データを扱うためのデータセットオブジェクトの作成を行う．


## データのダウンロード

プログラムに必要なデータをダウンロードします．

今回使用するデータは，Kaggleで公開されているBitcoinの価格を予測するデータセットです．

https://www.kaggle.com/datasets/team-ai/bitcoin-price-prediction


In [ ]:
import gdown
gdown.download('https://drive.google.com/uc?id=1_Gdneij6TP6CK_HCommCtbaitVfoY-fN', 'BitcoinPricePrediction.zip', quiet=False)
!unzip -q -o BitcoinPricePrediction.zip

ここで，一度データセットを確認してみましょう．

データ（フォルダ）を確認すると，BitcoinPricePredictionフォルダの中にTraining.csvとTest.csvという二つのCSVファイルが保存されています．

![BitcoinDataDir.png](https://qiita-image-store.s3.ap-northeast-1.amazonaws.com/0/143078/e81aa369-427e-fe22-03cf-03cd82c917ce.png)

### CSVファイルの中身

それぞれの中身を見ると，
* Date: 日付
* Open: 始値
* High: 最高値
* Low: 最安値
* Close: 終値
* Volume: 取引ボリューム（取引数量）
* Market Cap: 時価総額
という列があり，それぞれの日付で値を持っていることがわかります．

また，Dateの値を確認すると，新 --> 古の順番に日付が並んでいることがわかります．

今回は，「Open, High, Low, Close」の値から翌日の「High, Low」を予測する再帰型ニューラルネットワークを構築して学習してみましょう．

## モジュールのインポートとGPUの確認

はじめに必要なモジュールをインポートする．
また，GPUが使用可能かどうかを確認する．

In [ ]:
import os
import glob
import csv
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Use device:', device)

## データセットクラスの作成

ここでは，ダウンロードしたCSVファイルの形式に合わせて，PyTorchのデータセットクラスを自作します．

In [ ]:
class BitcoinPriceDataset(Dataset):

  def __init__(self, csv_file_path, time_window=10, max_price=None):
    super().__init__()

    self.csv_file_path = csv_file_path
    self.time_window = time_window

    ### csvファイルの読み込み
    with open(self.csv_file_path, 'r') as file:
      reader = csv.reader(file)

      ### 1行ずつデータをリストへ追加
      csv_data_source = []
      for row in reader:
        csv_data_source.append(row)

    ### ヘッダー行の保存（使用しないかもしれません）
    self.header = csv_data_source[0]
    ### データ行の保存（ヘッダ以外の行を保存）
    self.csv_data = csv_data_source[1:]

    ### 総データ数の保存
    self.num_data = len(self.csv_data)

    ### 日付データの保存（使用しないかもしれません）
    self.date = []
    for row in self.csv_data:
      self.date.append(row[0])

    ### 数値データの保存
    self.bitcoin_data = torch.zeros([self.num_data, 4], dtype=torch.float32)
    for i, row in enumerate(self.csv_data):
      self.bitcoin_data[i, 0] = float(row[1])
      self.bitcoin_data[i, 1] = float(row[2])
      self.bitcoin_data[i, 2] = float(row[3])
      self.bitcoin_data[i, 3] = float(row[4])

    ### データの順番を入れ替え（新~旧 --> 旧~新）
    self.date.reverse()
    self.bitcoin_data = torch.flipud(self.bitcoin_data)

    ### 最大の価格値
    if max_price is None:
      self.max_price = torch.max(self.bitcoin_data)
    else:
      self.max_price = max_price

    ### 数値データの正規化（0.0 ~ 1.0）
    self.bitcoin_data /= self.max_price

  def __getitem__(self, item):

    input = self.bitcoin_data[item:item+self.time_window, :]
    output = self.bitcoin_data[item+1:item+self.time_window+1, 1:3]

    return input, output

  def __len__(self):
    return self.num_data - self.time_window

## ネットワークモデルの定義

続いてネットワークを定義します．

In [ ]:
class MyRNN(nn.Module):

  ### ネットワーク構造
  # RNN (RNNCell) --> 全結合層 --> 予測結果

  def __init__(self, in_size=4, out_size=2, hidden_size=32):
    super().__init__()

    ### RNN層の定義
    self.recurrent = nn.RNNCell(input_size=in_size, hidden_size=hidden_size)

    ### 全結合層の定義
    self.fc = nn.Linear(in_features=hidden_size, out_features=out_size)

  def forward(self, x, hx):
    hx = self.recurrent(x, hx)
    h = self.fc(hx)
    return h, hx

## 学習の準備

ここでは，学習に必要な

* ネットワークモデル
* 誤差関数
* 最適化手法
* データセット

の定義を行います．

**DataLoaderのnum_workersについて**

`torch.utils.data.DataLoader`の引数である`num_workers`は，データを読み込んで準備する処理を並列処理するための引数です．例えば，`num_workers=10`とした場合には，10並列でデータの読込処理 (データセットクラスの`__getitem__()`) を10並列で実行してくれます．そのため，使用する計算機のCPU性能に合わせて，ある程度大きな数を指定しておくとデータの読込処理が早くなり，学習の高速化が期待できます．

In [ ]:
### ネットワークモデル
n_hidden = 32
model = MyRNN(in_size=4, out_size=2, hidden_size=n_hidden).to(device)

### 誤差関数
criterion = nn.MSELoss().to(device)

### 最適化関数
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

### データセットクラス
time_window = 10
train_dataset = BitcoinPriceDataset(csv_file_path="BitcoinPricePrediction/Training.csv", time_window=time_window, max_price=None)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

### 学習データ内の最大の価格の値を保存 (後程テストに使用します)
train_max_price = train_dataset.max_price

## 学習の開始

上で定義したモデルや最適化手法，データセットを用いて学習を行います．
エポックごとにモデルのパラメータを`checkpoint-{epoch}.pt`として保存し，評価の際に読み込みます．

In [ ]:
### Epoch数などの指定
num_epochs = 20

### ネットワークを学習モードへ変更
model.train()

### 学習経過を保存するためのリストを用意
loss_list = []

### 学習ループ (for文)
print("training; start ---------------------")
for epoch in range(1, num_epochs+1):
  print("Epoch:", epoch)

  # 1 epochごとの学習経過を計算するための変数を用意
  loss_sum = 0.0

  for input, label in train_loader:
    input = input.to(device)
    label = label.to(device)

    # 隠れ状態の変数の初期化
    hx = torch.zeros(input.size(0), n_hidden, device=device)

    loss = 0.0
    for time_index in range(input.size(1)):
      y, hx = model(input[:, time_index, :], hx)
      loss += criterion(y, label[:, time_index, :])

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # lossを加算（学習経過確認用）
    loss_sum += loss.item()

  ### epochが終了したタイミングでそのエポックの平均誤差を表示（1 epoch内のiterationとtime_windowで割った値）
  print("  Loss:", loss_sum / len(train_loader) / time_window)

  ### 上で表示した数値をリストへ保存
  loss_list.append(loss_sum / len(train_loader) / time_window)

  ### モデルパラメータの保存
  torch.save(model.state_dict(), "checkpoint-{:04d}.pt".format(epoch))

print("training; done ----------------------")

## 評価

学習したモデルを用いて評価を行います．

### モデルの読み込み

学習中に保存したモデルを読み込みます．

In [ ]:
### ネットワークモデルの定義（この時点ではパラメータはランダム）
model = MyRNN(in_size=4, out_size=2, hidden_size=n_hidden).to(device)

### 学習したモデルパラメータの読み込み
trained_parameter = torch.load("checkpoint-{:04d}.pt".format(num_epochs), map_location=device)
model.load_state_dict(trained_parameter)

### 推論と結果の表示

読み込んだ学習済みモデルを用いて予測を行います．

今回のネットワークは，数値データ（価格）の予測値を出力するモデルのため，誤差ではなく，その予測値をリストに保存して，後程グラフ表示をして結果を確認します．

**学習データに対する予測**

まずは，学習に使用したデータでどの程度予測できるかを確認します．

In [ ]:
pred = []
true = []

train_dataset_eval = BitcoinPriceDataset(csv_file_path="BitcoinPricePrediction/Training.csv", time_window=1, max_price=train_max_price)
train_loader_eval = torch.utils.data.DataLoader(train_dataset_eval, batch_size=1, shuffle=False, num_workers=2)

model.eval()

with torch.no_grad():

  # 隠れ状態の変数の初期化
  hx = torch.zeros(1, n_hidden, device=device)

  for input, label in train_loader_eval:
    input = input.to(device)
    label = label.to(device)

    output, hx = model(input[:, 0, :], hx)

    pred.append(output.tolist())
    true.append(label.tolist())

### 保存した正解・予測結果（リスト形式）をtorch.tensor形式に変換
pred_tensor = torch.tensor(pred).squeeze()
true_tensor = torch.tensor(true).squeeze()

### グラフの横軸用のリストを用意
time_index = list(range( pred_tensor.size(0) ))
print("time index list:", time_index)

### グラフの描画・表示・保存
plt.plot(time_index, pred_tensor[:, 0], '-b', label='high pred')
plt.plot(time_index, true_tensor[:, 0], '-r', label='high true')
plt.plot(time_index, pred_tensor[:, 1], '-c', label='low pred')
plt.plot(time_index, true_tensor[:, 1], '-y', label='low true')
plt.xlabel("day")
plt.ylabel("price")
plt.title("Prediction Results for Training Data")
plt.legend()
plt.savefig("prediction_train.pdf")
plt.show()
plt.clf()

**テストデータに対する予測**

次に，テスト用データでどの程度予測できるかを確認します．

In [ ]:
pred = []
true = []

test_dataset = BitcoinPriceDataset(csv_file_path="BitcoinPricePrediction/Test.csv", time_window=1, max_price=train_max_price)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

model.eval()

with torch.no_grad():

  # 隠れ状態の変数の初期化
  hx = torch.zeros(1, n_hidden, device=device)

  for input, label in test_loader:
    input = input.to(device)
    label = label.to(device)

    output, hx = model(input[:, 0, :], hx)

    pred.append(output.tolist())
    true.append(label.tolist())


### 保存した正解・予測結果（リスト形式）をtorch.tensor形式に変換
pred_tensor = torch.tensor(pred).squeeze()
true_tensor = torch.tensor(true).squeeze()

### グラフの横軸用のリストを用意
time_index = list(range( pred_tensor.size(0) ))
print("time index list:", time_index)

### グラフの描画・表示・保存
plt.plot(time_index, pred_tensor[:, 0], '-b', label='high pred')
plt.plot(time_index, true_tensor[:, 0], '-r', label='high true')
plt.plot(time_index, pred_tensor[:, 1], '-c', label='low pred')
plt.plot(time_index, true_tensor[:, 1], '-y', label='low true')
plt.xlabel("day")
plt.ylabel("price")
plt.title("Prediction Results for Test Data")
plt.legend()
plt.savefig("prediction_test.pdf")
plt.show()
plt.clf()

## 課題

1. `nn.RNNCell`を`nn.LSTMCell`や`nn.GRUCell`に変更して，予測精度がどのように変化するか確認してみましょう．
    * `LSTMCell`はセル状態`cx`も入出力する点に注意してください．
2. 予測対象を「High, Low」以外の値（例えば「Close」）に変更してみましょう．
3. `time_window`や`n_hidden`などのハイパーパラメータを変更して，予測精度がどのように変化するか確認してみましょう．